# Y Gate

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter
from io import BytesIO

import torch
from IPython.display import Image, SVG, display
from matplotlib import pyplot as plt

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import CliffordCall
from flagquantum.compiler.registers.primitives.pauli_gate import pauli_gate_ir
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score
from flagquantum.simulation.unitary import get_unitary

dtype = torch.complex128

## `YGate`
The Pauli Y gate.

This causes a bit flip (with a complex phase): Y|0> = 1j|1>, Y|1> = -1j|0>

#### Registers
 - `q`: The qubit.


In [ ]:
def show_score(name, ports, wire_symbol):
    widths = dict.fromkeys(ports, 1)
    layout = layout_register_score(widths, ports, (RegisterStep(name, ports, ports, name.lower()),))
    fig, _ = draw_register_score(register_score_data(layout, wire_symbol=wire_symbol))
    with BytesIO() as png:
        fig.savefig(png, format="png", bbox_inches="tight")
        display(Image(data=png.getvalue()))
    plt.close(fig)

### Example Instances

In [ ]:
y_gate = pauli_gate_ir("y", 0)
y_operation = FieldOperation(1, y_gate)
y = Circuit(1, dtype=dtype).append(y_operation)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("Y", Signature.build(q=1), {})])))

In [ ]:
def y_symbol(step, port, side):
    if port is None:
        return "Text", {"text": "", "fontsize": 10}
    return "TextBox", {"text": "Y" if step.name == "Y" else port}

show_score("Y", ("q",), y_symbol)

## `CYGate`
A controlled Y gate.

#### Registers
 - `ctrl`: The control qubit.
 - `target`: The target qubit.


In [ ]:
cy_signature = Signature.build(ctrl=1, target=1)

### Example Instances

In [ ]:
cy_gate = pauli_gate_ir("y", 1, controls=(0,), values=(1,))
cy_operation = FieldOperation(2, cy_gate)
cy = Circuit(2, dtype=dtype).append(cy_operation)
assert tuple(gate.name for gate in cy_gate) == ("cy",)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("CY", cy_signature, {})])))

In [ ]:
def cy_symbol(step, port, side):
    if port is None:
        return "Text", {"text": "", "fontsize": 10}
    if step.name == "CY":
        if port == "ctrl":
            return "Circle", {"filled": True}
        return "TextBox", {"text": "Y"}
    return "TextBox", {"text": port}

show_score("CY", ("ctrl", "target"), cy_symbol)

### Verify the complex action and resource counts

The controlled gate keeps the control wire and applies Y in its active sector.
Compare the complete matrices, including the signs of the imaginary amplitudes.

In [ ]:
expected_y = torch.tensor([[0, -1j], [1j, 0]], dtype=dtype)
expected_cy = torch.block_diag(torch.eye(2, dtype=dtype), expected_y)
torch.testing.assert_close(get_unitary(y), expected_y)
torch.testing.assert_close(get_unitary(cy), expected_cy)
torch.testing.assert_close(get_unitary(cy.append(cy_operation)), torch.eye(4, dtype=dtype))
for gates, call in ((y_gate, CliffordCall("y", 1)), (cy_gate, CliffordCall("cy", 2))):
    print("Emitted primitives", dict(Counter(gate.name for gate in gates)))
    print("Source cost model", query_qec_gate_cost(call))